# Treasure maze: how the game works, why training is slow, and how to make it fast

**What this notebook is for.** You are training a Keras network to play a maze game (a pirate looking for treasure), and your training run was far from 100% after 1.5 hours. This notebook (1) explains how the **game** works, line by line, so you can check your machine learning is applied correctly; (2) **measures** where the training time goes; and (3) gives you a **drop-in fast version** that reached a 100% win rate, and passed the "every starting cell wins" check, in about 3.5 minutes on my machine.

**Important caveat.** I have not seen your `TreasureMaze.py`, `GameExperience.py` or your own training code. I rebuilt the *standard* versions of those two classes (the ones in the widely used treasure-hunt template, which your imports match) to test everything here. This notebook imports **your** files, so when you run it you are testing your versions. If a number differs a lot from mine, or something errors, that tells you your files differ from the standard ones: tell me.

**Short version of the diagnosis** (measured in Part 3): in the standard `GameExperience.get_data`, every training step calls `model.predict` **twice per remembered sample, one sample at a time**. With a batch of 50 that is 100 separate `predict` calls, about **6 seconds per training step**. Add `fit(epochs=8)` and `evaluate` each step, and 1.5 hours is only a few hundred steps, or about 5 to 10 games of training. Your run was not learning too slowly; it simply had not had time to learn. The fix is to do the same maths in one batched call.

In [1]:
from __future__ import print_function
import os, sys, time, datetime, json, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # quiet TensorFlow's startup messages (optional)
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import clone_model
from keras.models import Sequential
from keras.layers import Dense, Activation, PReLU
from keras.optimizers import SGD , Adam, RMSprop
import matplotlib.pyplot as plt
from TreasureMaze import TreasureMaze
from GameExperience import GameExperience

print("imports worked")

imports worked


## Part 1: how the game works

The maze is a grid of numbers: `1.0` is a free cell, `0.0` is a wall. The pirate starts somewhere and must reach the **bottom-right cell** (the treasure). The agent can choose among four actions: `0` left, `1` up, `2` right, `3` down.

In [2]:
maze = np.array([
    [ 1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  0.,  1.,  1.,  1.,  0.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  0.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  0.,  0.,  0.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.]
])

qmaze = TreasureMaze(maze)
print("maze is", maze.shape, "with", maze.size, "cells")
print("treasure cell:", qmaze.target)
print("free cells the pirate can start from:", len(qmaze.free_cells))
print("a game is lost when total reward drops below", qmaze.min_reward)

maze is (8, 8) with 64 cells
treasure cell: (7, 7)
free cells the pirate can start from: 50
a game is lost when total reward drops below -32.0


**What each line made.** `maze` is a numpy array of shape `(8, 8)`: 64 numbers. `TreasureMaze(maze)` builds the game object. It lists the **free cells** (cells equal to 1.0) except the treasure cell, because the pirate will be started from a random one of these each game. `min_reward` is the "give up" line: the game is **lost** as soon as the pirate's *total* reward falls below `-0.5 x maze.size = -32`. That stops the pirate from wandering forever.

In [3]:
qmaze.reset((0, 0))                       # put the pirate at the top-left cell
state = qmaze.observe()                   # what the neural network is shown

print("type :", type(state).__name__)
print("shape:", state.shape, "  <- one row of 64 numbers (a batch of one)")
print(state.reshape(8, 8))                # the same 64 numbers laid out as the maze
print("valid actions at (0, 0):", qmaze.valid_actions(), " (0 left, 1 up, 2 right, 3 down)")

type : ndarray
shape: (1, 64)   <- one row of 64 numbers (a batch of one)
[[0.5 0.  1.  1.  1.  1.  1.  1. ]
 [1.  0.  1.  1.  1.  0.  1.  1. ]
 [1.  1.  1.  1.  0.  1.  0.  1. ]
 [1.  1.  1.  0.  1.  1.  1.  1. ]
 [1.  1.  0.  1.  1.  1.  1.  1. ]
 [1.  1.  1.  0.  1.  0.  0.  0. ]
 [1.  1.  1.  0.  1.  1.  1.  1. ]
 [1.  1.  1.  1.  0.  1.  1.  1. ]]
valid actions at (0, 0): [3]  (0 left, 1 up, 2 right, 3 down)


**What the network sees.** `observe()` draws the maze as an 8 x 8 picture, then **flattens it into one row of 64 numbers** (shape `(1, 64)`: one row because Keras expects a *batch*, and here the batch holds one picture). Walls are `0.0`, free cells are `1.0`, and the pirate's cell is `0.5`. That is the whole input: the network sees the walls and where the pirate is. It does **not** see which cells the pirate has already visited (the visited marks are erased when the picture is drawn), so it must learn a rule of the form "when the pirate is *here*, go *there*".

Compare with the CartPole and maze notebooks: this 64-number vector plays the role of the "state", and the network replaces the Q-table. The output is four numbers, one Q value per action, exactly like a row of the table.

`valid_actions()` lists the moves that do not hit a wall or the edge. At `(0, 0)` the cell to the right is a wall, so only "down" (`3`) is valid.

In [4]:
qmaze.reset((0, 0))
moves = [(3, "down"), (3, "down"), (1, "up"), (2, "right (a wall)"), (3, "down")]
for action, name in moves:
    envstate, reward, status = qmaze.act(action)
    row, col, mode = qmaze.state
    print(f"{name:16s} -> pirate at {(row, col)}, mode {mode:8s} reward {reward:6.2f}, total {qmaze.total_reward:6.2f}, status {status}")

down             -> pirate at (1, 0), mode valid    reward  -0.04, total  -0.04, status not_over
down             -> pirate at (2, 0), mode valid    reward  -0.04, total  -0.08, status not_over
up               -> pirate at (1, 0), mode valid    reward  -0.25, total  -0.33, status not_over
right (a wall)   -> pirate at (1, 0), mode invalid  reward  -0.25, total  -0.58, status not_over
down             -> pirate at (2, 0), mode valid    reward  -0.25, total  -0.83, status not_over


**The reward rules, from the output.** `qmaze.act(action)` moves the pirate and returns three things: the new picture (`envstate`), the **reward**, and the game **status** (`'not_over'`, `'win'` or `'lose'`). The rewards in the standard game are:

| What happened | Reward |
|---|---|
| Reached the treasure | **+1.0** |
| An ordinary move to a new cell | -0.04 |
| Moved onto a cell already visited | -0.25 |
| Tried a move into a wall or off the grid ("invalid") | -0.75 |
| Pirate has no valid moves ("blocked") | game lost at once |

Check this against the printed lines. The first two moves are ordinary (-0.04 each). Moving back up to a visited cell costs -0.25. The fourth line is the interesting one: pressing right into a wall is an **invalid** move, yet it cost -0.25, not -0.75. The reason is the *order* of the checks in the game: it asks "is the pirate standing on an already-visited cell?" before it asks "was the move invalid?", and the pirate was standing on a visited cell. On a fresh cell the same wall bump would cost -0.75. Knowing details like this is exactly why it helps to read the game's code. The structure is the same as the earlier maze notebook as the earlier maze notebook: a small cost per step so short routes win, and a bigger penalty for bad moves. The `total` column is what is compared with `min_reward`.

## Part 2: what the replay memory (`GameExperience`) does

`GameExperience` is the **replay memory** from the deep Q-learning section of the maze notebook. Every step, the training code hands it one experience `[state, action, reward, next_state, game_over]`. Training then samples a batch of old experiences and turns each into a **training example**: an input (the state) and a **target** (what the network should have predicted). Let us compute one target by hand.

In [5]:
random.seed(0); np.random.seed(0); tf.random.set_seed(0)

def build_model(maze, lr=0.001):
    model = Sequential()
    model.add(Dense(maze.size, input_shape=(maze.size,)))   # 64 inputs -> 64 neurons
    model.add(PReLU())                                      # PReLU: a ReLU whose negative slope is learned
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(4))                                     # 4 outputs: one Q value per action
    model.compile(optimizer="adam", loss="mse")
    return model

model = build_model(maze)
exp = GameExperience(model, max_memory=1000)

qmaze.reset((0, 0)); env = qmaze.observe()
for _ in range(40):                                         # play 40 random moves and remember them
    action = random.choice(qmaze.valid_actions())
    nxt, reward, status = qmaze.act(action)
    over = status != "not_over"
    exp.remember([env, action, reward, nxt, over])
    env = nxt
    if over:
        qmaze.reset((0, 0)); env = qmaze.observe()

envstate, action, reward, envstate_next, game_over = exp.memory[10]
q_now = exp.predict(envstate)
q_next = exp.predict(envstate_next)
print("memory holds", len(exp.memory), "experiences; discount =", exp.discount)
print("action taken:", action, "| reward:", reward, "| game over:", game_over)
print("network's Q values for the state :", np.round(q_now, 3))
print("network's Q values for next state:", np.round(q_next, 3))
target = reward if game_over else reward + exp.discount * np.max(q_next)
print("target for that action = reward + discount * best next Q =", round(float(target), 4))

memory holds 40 experiences; discount = 0.95
action taken: 1 | reward: -0.25 | game over: False
network's Q values for the state : [ 0.309 -0.531  0.118 -0.369]
network's Q values for next state: [ 0.312 -0.475  0.221 -0.459]
target for that action = reward + discount * best next Q = 0.0467


**What this shows.** The network has not been trained, so its Q values are small meaningless numbers. The target for the action taken is computed with the **same rule as the maze notebook**: `target = reward + discount x (best Q value of the next state)`, or just `reward` if the game ended. `discount` is 0.95 (the gamma from before). The target vector for that training example is then the network's *current* prediction for all four actions, with **only the chosen action's entry replaced** by this target. That way the error is zero for the actions we did not try, and only the tried action gets corrected.

Now the important part: look at **how `get_data` computes that for 50 samples**. In the standard version it loops over the samples and, for each, calls `self.predict(envstate)` and `self.predict(envstate_next)`: **two separate `model.predict` calls per sample**. That is the slow part, as the next cell shows.

## Part 3: measure where the time goes

In [6]:
def timeit(label, f, n=5):
    f()                                                      # warm-up call, not timed
    t = time.time()
    for _ in range(n):
        f()
    print(f"{label:48s} {(time.time() - t) / n * 1000:9.1f} ms")

inputs, targets = exp.get_data(30)
timeit("one model.predict (inside GameExperience)", lambda: exp.predict(env))
timeit("get_data(30)  (2 predicts per sample = 60 calls)", lambda: exp.get_data(30), n=2)
timeit("model.fit(epochs=8, batch_size=16)", lambda: model.fit(inputs, targets, epochs=8, batch_size=16, verbose=0), n=3)
timeit("model.evaluate(inputs, targets)", lambda: model.evaluate(inputs, targets, verbose=0))
timeit("model.train_on_batch(inputs, targets)", lambda: model.train_on_batch(inputs, targets))
timeit("model(x, training=False)  (direct call)", lambda: model(env, training=False))

one model.predict (inside GameExperience)             57.8 ms
get_data(30)  (2 predicts per sample = 60 calls)    3631.1 ms
model.fit(epochs=8, batch_size=16)                   525.3 ms
model.evaluate(inputs, targets)                       57.5 ms
model.train_on_batch(inputs, targets)                  1.3 ms
model(x, training=False)  (direct call)                3.0 ms


**Reading the timings** (these are from my machine; yours will differ, but the *ratios* should look similar). A single `model.predict` costs tens of milliseconds, because `predict` is built for large datasets and has a lot of fixed overhead on every call. `get_data` makes two such calls per sample, so it costs about 3.7 seconds for 30 samples in the run above (and about 6 seconds for 50 in my earlier test). Compare that with the actual learning work: `train_on_batch` (one gradient step on the whole batch) takes about a millisecond, and a direct call `model(x)` is about 25 times cheaper than `predict`.

So in the standard training loop, a **single training step is dominated by ~100 tiny `predict` calls**, with `fit(epochs=8)` and `evaluate` adding more. A game is often over 100 steps long (random play wanders for a long time before it loses), and you need **hundreds of games**. That is days of waiting, which matches what you saw.

## Part 4: the same maths, in one batch

We do not need to change what is computed, only how. Put all the sampled states and all the next states into **two arrays**, push them through the network **in one call**, and then use ordinary numpy to build the targets. The result is the same set of targets, up to tiny floating-point differences.

We **subclass** `GameExperience` (so we do not edit the provided file) and replace just `predict` and `get_data`.

In [7]:
class FastExperience(GameExperience):
    def predict(self, envstate):
        # direct call: much less overhead than model.predict for a single input
        return self.model(envstate, training=False).numpy()[0]

    def get_data(self, data_size=10):
        mem_size = len(self.memory)
        data_size = min(mem_size, data_size)
        idx = np.random.choice(mem_size, data_size, replace=False)    # which memories to use
        batch = [self.memory[j] for j in idx]

        states  = np.vstack([b[0] for b in batch])                    # shape (data_size, 64)
        nexts   = np.vstack([b[3] for b in batch])                    # shape (data_size, 64)
        actions = np.array([b[1] for b in batch])                     # shape (data_size,)
        rewards = np.array([b[2] for b in batch], dtype=float)
        over    = np.array([b[4] for b in batch], dtype=float)        # 1.0 if the game ended

        both = self.model(np.vstack([states, nexts]), training=False).numpy()   # ONE call for everything
        targets = both[:data_size].copy()                             # predictions for the states
        q_next = both[data_size:].max(axis=1)                         # best Q of each next state

        # only the action taken gets a new value: reward (+ discounted future unless the game ended)
        targets[np.arange(data_size), actions] = rewards + self.discount * q_next * (1 - over)
        return states, targets

fast = FastExperience(model, max_memory=1000)
fast.memory = exp.memory                       # same memories as the original object

np.random.seed(123); X_old, Y_old = exp.get_data(20)
np.random.seed(123); X_new, Y_new = fast.get_data(20)
print("same inputs :", np.allclose(X_old, X_new))
print("largest difference between the two sets of targets:", float(np.abs(Y_old - Y_new).max()))

inputs, targets = exp.get_data(30)
timeit("ORIGINAL get_data(30)", lambda: exp.get_data(30), n=2)
timeit("FAST get_data(30)", lambda: fast.get_data(30), n=20)

same inputs : True
largest difference between the two sets of targets: 1.4901161193847656e-08
ORIGINAL get_data(30)                               3584.6 ms
FAST get_data(30)                                      4.2 ms


**What each part does.**
- `np.vstack([...])` stacks a list of `(1, 64)` rows into one `(data_size, 64)` array. That is the *batch*.
- One call to `self.model(...)` on the states stacked on top of the next states returns all the predictions at once; the first `data_size` rows belong to the states and the rest to the next states.
- `both[data_size:].max(axis=1)` is the best next Q value **per row** (this is the `max` and `axis=1` you met in the numpy primer).
- `targets[np.arange(data_size), actions] = ...` is the **fancy indexing** from the neural-network section: it overwrites exactly one number per row, the entry for the action taken.
- `(1 - over)` is 0 for finished games, so there is no future value after the game ends. This is the "no future after the goal" rule again.

**The equivalence check.** Seeding numpy the same way before each call makes both versions sample the *same* memories. The inputs are identical and the targets differ only by a tiny rounding amount (the printed largest difference, about `1e-8`), which shows the fast version computes the same targets. The last two lines compare speed: in my run the original `get_data(30)` took about 3.8 seconds and the fast one about 3 milliseconds, roughly **a thousand times faster**.

## Part 5: a complete fast training loop

This is the standard training loop with four changes (marked `# CHANGED`): the fast experience class, `train_on_batch` instead of `fit` plus `evaluate`, a few gradient steps per game step, and a fast play-through for the completion check. It stops when the pirate wins 32 games in a row **and** wins from every free starting cell, the usual 100% test.

In [ ]:
def play_game(model, qmaze, pirate_cell):
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    while True:
        q = model(envstate, training=False).numpy()               # CHANGED: direct call, not model.predict
        action = int(np.argmax(q[0]))
        envstate, reward, game_status = qmaze.act(action)
        if game_status == "win":
            return True
        elif game_status == "lose":
            return False

def completion_check(model, qmaze):
    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            return False
        if not play_game(model, qmaze, cell):
            return False
    return True

def qtrain(model, maze, n_epoch=3000, max_memory=1000, data_size=50,
           grad_steps=4, epsilon=0.1, time_limit=1800):
    qmaze = TreasureMaze(maze)
    experience = FastExperience(model, max_memory=max_memory)     # CHANGED: fast experience class
    win_history = []
    hsize = qmaze.maze.size // 2                                  # window of recent games: 32
    win_rate = 0.0
    start = time.time()
    for epoch in range(n_epoch):
        agent_cell = random.choice(qmaze.free_cells)              # random start cell each game
        qmaze.reset(agent_cell)
        envstate = qmaze.observe()
        game_over = False
        while not game_over:
            valid_actions = qmaze.valid_actions()
            if not valid_actions:
                break
            prev_envstate = envstate
            if np.random.rand() < epsilon:                        # explore
                action = random.choice(valid_actions)
            else:                                                 # exploit
                action = int(np.argmax(experience.predict(prev_envstate)))
            envstate, reward, game_status = qmaze.act(action)
            if game_status == "win":
                win_history.append(1); game_over = True
            elif game_status == "lose":
                win_history.append(0); game_over = True
            experience.remember([prev_envstate, action, reward, envstate, game_over])
            inputs, targets = experience.get_data(data_size=data_size)
            for _ in range(grad_steps):                           # CHANGED: train_on_batch, no fit/evaluate
                loss = model.train_on_batch(inputs, targets)
        if len(win_history) > hsize:
            win_rate = sum(win_history[-hsize:]) / hsize
        if epoch % 20 == 0:
            print(f"epoch {epoch:4d} | win rate {win_rate:.3f} | epsilon {epsilon:.2f} | {time.time() - start:6.1f}s")
        if win_rate > 0.9:
            epsilon = 0.05
        if len(win_history) >= hsize and sum(win_history[-hsize:]) == hsize and completion_check(model, qmaze):
            print(f"Reached 100% win rate at epoch {epoch}, after {time.time() - start:.1f} seconds")
            break
        if time.time() - start > time_limit:
            print("Stopped: time limit reached")
            break
    return win_history

random.seed(0); np.random.seed(0); tf.random.set_seed(0)
model = build_model(maze)
win_history = qtrain(model, maze)

**What my run printed** (about 3.5 minutes on a laptop-class CPU; yours will differ in time and, since training involves randomness, in the exact epochs):

```text
epoch    0 | win rate 0.000 | epsilon 0.10 | 2.8s
epoch   20 | win rate 0.000 | epsilon 0.10 | 35.8s
epoch   40 | win rate 0.375 | epsilon 0.10 | 57.0s
epoch   60 | win rate 0.594 | epsilon 0.10 | 75.3s
epoch   80 | win rate 0.875 | epsilon 0.10 | 81.8s
epoch  100 | win rate 1.000 | epsilon 0.05 | 97.6s
epoch  120 | win rate 1.000 | epsilon 0.05 | 131.4s
epoch  140 | win rate 1.000 | epsilon 0.05 | 148.8s
epoch  160 | win rate 1.000 | epsilon 0.05 | 177.1s
Reached 100% win rate at epoch 177, after 213.5 seconds
```

**How to read that.** For the first ~30 epochs the pirate rarely wins, because early play is mostly random and games end in a loss. Then the win rate climbs steadily, which is the same wave of learning you saw in the maze notebook, now carried by the network instead of the table. At epoch ~100 the recent win rate is 100%, but the run keeps going for ~75 more epochs. Winning 32 games in a row from *random* starts is not enough: `completion_check` must also win from **every single free starting cell**, and a few awkward cells took longer to master. The run ends only when all of them pass.

## Part 6: how to make sure your own run is correct

1. **Time one training step** in your own loop (wrap the body in `time.time()`). If a single step takes more than about 50 ms, something in it is doing per-sample `predict` calls, `fit(epochs=8)` or `evaluate`. Those three are the usual culprits.
2. **Print progress every ~20 epochs** with the win rate *and elapsed time*, so you can see whether it is rising and how fast.
3. **Check epsilon.** It should start around 0.1 and drop to about 0.05 once the win rate passes 0.9. A much larger epsilon keeps the pirate wandering (and losing); zero stops it exploring.
4. **Check the stopping rule.** 100% means the last `maze.size // 2` games were all wins **and** `completion_check` passes. Plot `win_history` or the win rate to show the rise.
5. **Keep the maths the same.** The target for the action taken is `reward + discount x max Q(next)`, or just `reward` if the game ended. The fast version changes how it is computed, not what.
6. **Check your assignment's rules** about editing the provided `.py` files. Subclassing `GameExperience` inside your own notebook (as here) leaves the provided files untouched.

**If your result still isn't 100% after this,** the problem is not speed. Send me your `build_model`, your `qtrain` and your settings (epsilon, `max_memory`, `data_size`, learning rate, `n_epoch`), and I will check them line by line.